# 采样方法

> Sampling 是 AI 探索可能性空间的方式。

**类型：** 动手实现
**语言：** Python
**前置要求：** Phase 1, Lesson 06-07
**时间：** ~120 分钟

## 术语对照

- inverse CDF，逆累积分布函数
- rejection sampling，拒绝采样
- importance sampling，重要性采样
- Monte Carlo，蒙特卡洛
- MCMC，马尔可夫链蒙特卡洛
- Gibbs sampling，Gibbs 采样
- temperature sampling，温度采样
- top-k sampling，Top-K 采样
- top-p sampling，Top-P 采样
- reparameterization trick，重参数化技巧
- Gumbel-Softmax，Gumbel-Softmax
- diffusion sampling，扩散采样
## 关键术语

| 术语 | 含义 |
|------|------|
| Sampling | 按概率分布生成值。所有生成式 AI 背后的机制 |
| Inverse CDF | F_inverse(U) 将 uniform 样本转换为任意分布样本 |
| Rejection sampling | 从简单提议分布采样，以 target/proposal 比概率接受 |
| Importance sampling | 用 q(x) 的样本加权估计 p(x) 下的期望。PPO 核心 |
| Monte Carlo | 用样本平均近似积分。误差 O(1/√N)，与维度无关 |
| MCMC | 构造以目标为平稳分布的 Markov chain。Metropolis-Hastings 是基础算法 |
| Gibbs sampling | 逐变量条件采样。100% 接受率 |
| Temperature | T<1 锐化，T>1 平坦化 softmax 前的 logits |
| Top-p (nucleus) | 保留累积概率 ≥ p 的最少 token。候选集大小自适应 |
| Reparameterization trick | z=μ+σ·ε，将随机性移到无参 ε。使 VAE 可微 |
| Gumbel-Softmax | 对离散类别采样的可微近似。神经架构搜索和离散 VAE |
| Diffusion sampling | 从噪声开始，迭代应用学到的去噪步骤生成数据 |

## 学习目标

- 仅用 uniform 随机数从零实现 inverse CDF、rejection 和 importance sampling
- 构建语言模型 token 生成的 temperature、top-k 和 top-p（nucleus）sampling
- 解释 reparameterization trick 及其为何使 VAE 中的 backpropagation 通过 sampling 成为可能
- 运行 Metropolis-Hastings MCMC 从未归一化的目标分布中采样

## 问题

语言模型处理完你的 prompt，输出一个 50000 个 logits 的 vector。现在它必须选一个。怎么选？

总挑最高概率的 token，每次回答都一样。确定、枯燥。完全随机选，输出是乱码。答案在两者之间的某处，由 sampling 控制。

Sampling 不限于文本生成。RL 通过采样轨迹估计 policy gradient。VAE 通过从学到的分布中采样并反向传播来学习。Diffusion 模型通过采样噪声并迭代去噪生成图像。Monte Carlo 方法估计无解析解的积分。MCMC 算法探索不可能枚举的高维 posterior 分布。

每个生成式 AI 系统都是一个 sampling 系统。Sampling 策略决定输出的质量、多样性和可控性。

## 概念

### 四种核心角色

- **生成：** 语言模型、diffusion、GANs。Temperature/top-k/nucleus 是工程师每日调节的旋钮。
- **训练：** SGD 采样 mini-batch。Dropout 采样 neuron 停用。Importance sampling 在 PPO/TRPO 中降低 gradient 方差。
- **估计：** Monte Carlo 通过样本平均近似期望 loss、partition function、Bayesian evidence。
- **探索：** MCMC 探索 posterior。进化策略采样参数扰动。Thompson sampling 平衡探索与利用。

### Inverse CDF Method


算法：
  1. 生成 u ~ Uniform(0, 1)
  2. 返回 F_inverse(u)

为何有效：P(X ≤ x) = P(F_inverse(U) ≤ x) = P(U ≤ F(x)) = F(x)

指数分布：
  CDF: F(x) = 1 - exp(-λx)
  Inverse: x = -ln(u) / λ


离散版：构建 CDF 为累积和，生成 U，找第一个累积和超过 U 的索引。

### Rejection Sampling

目标分布 p(x)（可求值），提议分布 q(x)（可采样），界 M 使 p(x) ≤ M·q(x)。


1. x ~ q(x)
2. u ~ Uniform(0, 1)
3. 若 u < p(x) / (M·q(x))，接受 x
4. 否则拒绝，回到步骤 1

接受率 = 1/M


低维（1-3d）效果好。高维下接受率指数级下降。

### Importance Sampling


目标：估计 E_p[f(x)]

E_p[f(x)] = E_q[f(x) · w(x)]，其中 w(x) = p(x)/q(x)

估计量 ≈ (1/N) · Σ f(x_i) · w(x_i)，x_i ~ q(x)


PPO 的核心：用旧策略 π_old 收集轨迹，用 importance weight π_new/π_old 为新策略优化。

### Monte Carlo 估计

通过平均随机样本近似积分。误差 O(1/√N)，与维度无关。高维中无替代方案。

### Metropolis-Hastings MCMC

构造 Markov chain，其平稳分布为目标分布 p(x)。


1. x_0 起始
2. For t = 1, 2, ..., T:
   a. 提议 x' ~ q(x'|x_t)
   b. 接受比 α = [p(x')·q(x_t|x')] / [p(x_t)·q(x'|x_t)]
   c. 以概率 min(1, α) 接受：x_{t+1} = x'
      否则 x_{t+1} = x_t
3. 丢弃前 B 个样本（burn-in）
4. 返回剩余样本


对称提议下 α 简化为 p(x')/p(x)。Detail balance 保证 p(x) 是平稳分布。

实践要点：burn-in 丢弃早期样本，thinning 每 k 步保留一个，proposal scale 影响混合速度（高维下 Gaussian proposal 最优接受率 ≈ 0.234）。

### Gibbs Sampling

逐变量更新而非全体更新。需要能从每个条件分布 p(x_i | x_{-i}) 中采样。接受率 100%。变量高度相关时混合慢。

### Temperature Sampling（LLM 使用）


p_i = exp(z_i / T) / Σ exp(z_j / T)

T=1.0：标准 softmax
T→0：argmax（确定，总选最高 logit）
T→∞：uniform（全等可能）
T<1：锐化分布（更自信，更多样）


实践：T=0.0 适合事实 Q&A，0.3-0.7 代码生成，0.7-1.0 通用对话，1.0-1.5 创意写作。

### Top-k Sampling

仅保留概率最高的 k 个 token，重归一化，从中采样。k=40 典型。问题：k 固定。模型自信时（一个 token 95%）仍保留 39 个替代。

### Top-p (Nucleus) Sampling

保留最小 token 集合使其累积概率 ≥ p。p=0.9 典型。动态调整候选集大小。自信时（2-3 个），不确定时（200 个）。普遍优于 top-k。

### Reparameterization Trick（VAE 使用）


标准 sampling（不可微）：
  z ~ N(μ, σ²)  随机性阻断 gradient 流动

重参数化 sampling：
  ε ~ N(0, 1)         固定随机噪声，无参数
  z = μ + σ · ε       μ 和 σ 的确定可微函数

  dz/dμ = 1，dz/dσ = ε。Gradient 流过 μ 和 σ。


没有这个 trick，VAE 无法用标准 backpropagation 训练。这个单一洞见使 VAE 变得实用。

### Gumbel-Softmax

类别重参数化。Gumbel-Max trick（不可微）：对每个类别加 Gumbel 噪声，取 argmax。Gumbel-Softmax：用 softmax（带温度 τ）替代 hard argmax。τ→0 接近 one-hot。训练时前向用 hard argmax，反向用 soft Gumbel-Softmax gradient（straight-through estimator）。

应用：离散 VAE、神经架构搜索、hard attention 机制、离散 RL。

### Stratified Sampling

将采样空间分层，每层强制采样一个点。方差 ≤ 标准 Monte Carlo。函数光滑时改进最大。

### 与 Diffusion 模型的联系

Forward process 加噪声：x_t = √α_t·x_{t-1} + √(1-α_t)·ε。Reverse process 学去噪。每个去噪步骤是用 reparameterization trick 的采样步骤。整个图像生成过程是迭代采样。

## 动手实现


In [ ]:
import math, random

# Inverse CDF: 指数分布
def sample_exponential(lam):
    return -math.log(random.random()) / lam

# Rejection sampling
def rejection_sample(target_pdf, proposal_sample, proposal_pdf, M):
    while True:
        x = proposal_sample()
        if random.random() < target_pdf(x) / (M * proposal_pdf(x)):
            return x

# Importance sampling
def importance_sampling(f, target_pdf, proposal_pdf, proposal_sample, n):
    total = sum(f(proposal_sample()) * target_pdf(x)/proposal_pdf(x) for _ in range(n))
    return total / n

# Monte Carlo π
def monte_carlo_pi(n):
    inside = sum(1 for _ in range(n)
                 if random.uniform(-1,1)**2 + random.uniform(-1,1)**2 <= 1)
    return 4 * inside / n

# Metropolis-Hastings
def metropolis_hastings(target_log_pdf, proposal_sample, proposal_log_pdf,
                        x0, n_samples, burn_in):
    x, samples = x0, []
    for i in range(n_samples + burn_in):
        x_new = proposal_sample(x)
        log_alpha = (target_log_pdf(x_new) + proposal_log_pdf(x, x_new)
                     - target_log_pdf(x) - proposal_log_pdf(x_new, x))
        if math.log(random.random()) < log_alpha:
            x = x_new
        if i >= burn_in: samples.append(x)
    return samples

# Temperature sampling
def temperature_sample(logits, T):
    scaled = [z / T for z in logits]
    probs = softmax(scaled)
    return sample_categorical(probs)

# Top-p (nucleus) sampling
def top_p_sample(logits, p):
    probs = softmax(logits)
    indexed = sorted(enumerate(probs), key=lambda x: -x[1])
    cumsum, selected = 0, []
    for idx, prob in indexed:
        cumsum += prob; selected.append((idx, prob))
        if cumsum >= p: break
    sel_probs = [pr/sum(p for _, p in selected) for _, pr in selected]
    return selected[sample_categorical_idx(sel_probs)][0]

# Reparameterization trick
def reparam_sample(mu, sigma):
    return mu + sigma * random.gauss(0, 1)


完整实现见 `code/sampling.py`。
